In [2]:
import json
import string
import matplotlib.pyplot as plt
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted
from typing import List, Dict


%matplotlib inline

In [3]:
SENTIMENT_DICT = {
    "מר": -1, "מלוח מדי": -1, "יבש": -2, "שרוף": -3, "לא טעים": -2,
    "מועך מדי": -1, "קשה ללעיסה": -2, "חומצי מדי": -1, "משעמם": -1,
    "מיותר": -1, "חסר טעם": -2, "מושחת": -3, "מעופש": -3, "שומני מדי": -2,
    "חסר איזון": -2, "פריך": 1, "רך": 1, "רגיל": 1, "בסדר": 1,
    "מאוזן": 1, "קלאסי": 1, "נעים": 1, "סטנדרטי": 1, "פשוט": 1,
    "שגרתי": 1, "חביב": 1, "טעמו סביר": 1, "טעים": 2, "שווה": 2,
    "מענג": 2, "מוצלח": 2, "מלא טעם": 2, "מרענן": 2, "מעולה": 2,
    "משובח": 2, "קרמי": 2, "טעמו נהדר": 2, "מושקע": 2, "עשיר": 2,
    "איכותי": 2, "מושלם": 3, "מדהים": 3, "מופלא": 3, "מצוין": 3,
    "לא יאמן": 3, "מדהים בטעם": 3, "שובב": 3, "מושלם בטעמים": 3,
    "סופר טעים": 3, "מעורר השראה": 3, "אדיר": 3, "מדהים ביותר": 3
}

def calculate_sentiment(text: str) -> int:
    """מחשב ציון רגש על בסיס מילון מילים."""
    text_clean = text.translate(str.maketrans('', '', string.punctuation))
    words = text_clean.split()
    score = 0
    for w in words:
        if w in SENTIMENT_DICT:
            score += SENTIMENT_DICT[w]
    return score

In [4]:
def load_json(filename: str = "conversation.json") -> List[Dict]:
    """טוען את היסטוריית השיחות מקובץ JSON."""
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def save_to_json(conversation: Dict, filename: str = "conversation.json"):
    """שומר שיחה חדשה לתוך קובץ ה-JSON."""
    history = load_json(filename)
    history.append(conversation)
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

In [5]:
class ServiceManager:
    def __init__(self):
        
        self.api_key = "input your own api key"
        genai.configure(api_key=self.api_key)
        self.model = genai.GenerativeModel("models/gemini-flash-latest")

    def question_and_answers(self, prompt: str) -> str:
        try:
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as ex:
            print(f"שגיאה בתקשורת: {ex}")
            return None

    def interaction_with_bot(self):
        history = load_json()
        current_conversation = {"id": len(history) + 1, "messages": []}
        
        print("צ'אט פעיל (הקלידי 'סיום' כדי לעצור ולשמור):")
        while True:
            user_input = input("את: ").strip()
            if user_input.lower() == "סיום":
                break
            
            answer = self.question_and_answers(user_input)
            if answer:
                print(f"בוט: {answer}")
                current_conversation["messages"].append({"user": user_input, "model": answer})
        
        if current_conversation["messages"]:
            save_to_json(current_conversation)
            print("השיחה נשמרה בהצלחה.")

In [ ]:

service = ServiceManager()
service.interaction_with_bot()

In [ ]:
def run_full_analysis():
    conversations = load_json()
    if not conversations:
        print("לא נמצאו נתונים לניתוח.")
        return

    last_conv = conversations[-1]
    scores = []
    
    print(f"\n--- פירוט סנטימנט לשיחה {last_conv['id']} ---")
    print(f"{'הודעה':<5} | {'ציון':<5} | {'תוכן'}")
    print("-" * 40)
    
    for i, msg in enumerate(last_conv["messages"], 1):
        score = calculate_sentiment(msg["user"])
        scores.append(score)
        print(f"{i:<5} | {score:<5} | {msg['user'][:40]}...")

   
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(scores) + 1), scores, marker='o', color='blue', label='Sentiment')
    plt.axhline(0, color='red', linestyle='--', alpha=0.5)
    plt.title("Sentiment Flow Chart")
    plt.xlabel("Message Number")
    plt.ylabel("Score")
    plt.grid(True)
    plt.legend()
    plt.show()


run_full_analysis()